# K-Means Clustering Example (Mall Customers Dataset)

**Goal: Discover distinct customer segments based on Age, Annual Income, and Spending Score.**

- **Samples:** 200 mall shoppers  |  **Features:** 3  |  **Labels:** None

## 1. Setup and Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

# Find the repo root reliably on any machine.
# We search upward from the current working directory until we find
# the 'data' folder, which only exists at the repo root.
def _find_repo_root():
    path = os.path.abspath(os.getcwd())
    for _ in range(10):  # search up to 10 levels up
        if os.path.isdir(os.path.join(path, 'data')) and os.path.isdir(os.path.join(path, 'src')):
            return path
        path = os.path.dirname(path)
    raise RuntimeError(
        "Could not find repo root. Make sure you launched Jupyter from inside the CMOR-438 folder."
    )

REPO_ROOT = _find_repo_root()

# Algorithm source files live in src/unsupervised/ at the repo root
SRC_UNSUP = os.path.join(REPO_ROOT, 'src', 'unsupervised')
sys.path.insert(0, SRC_UNSUP)

# Data files live in data/ at the repo root
DATA_DIR = os.path.join(REPO_ROOT, 'data')
from k_means_clustering import KMeans
from sklearn.preprocessing import StandardScaler

mall = pd.read_csv(os.path.join(DATA_DIR, 'Mall_Customers.csv'))
MALL_FEATURES = ['Age', 'Annual Income (k$)', 'Spending Score (1-100)']
print(f"Dataset loaded: {mall.shape[0]} samples, {len(MALL_FEATURES)} features.")
print(mall[MALL_FEATURES].describe().round(1).to_string())

## 2. Preprocessing

Standardise features — K-Means uses Euclidean distance so scale matters.

In [ ]:
X_raw = mall[MALL_FEATURES].values.astype(float)
scaler = StandardScaler().fit(X_raw)
X = scaler.transform(X_raw)
print(f"Standardised — mean: {X.mean(axis=0).round(3)}, std: {X.std(axis=0).round(3)}")

## 3. Choose the Best k

Elbow method and silhouette score to select the optimal number of clusters.

In [ ]:
inertias, silhouettes = [], []
k_range = range(2, 11)
for k in k_range:
    km = KMeans(k=k, init='k-means++', n_init=5, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(km.silhouette_score(X))

best_k = list(k_range)[np.argmax(silhouettes)]
print(f'Best K by silhouette: {best_k}  (score={max(silhouettes):.4f})')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(list(k_range), inertias, 'o-', color='steelblue', lw=1.5, ms=6)
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('K-Means - Elbow Method', fontweight='bold')
axes[1].plot(list(k_range), silhouettes, 's-', color='darkorange', lw=1.5, ms=6)
axes[1].axvline(best_k, color='red', linestyle='--', lw=1.2, label=f'Best k={best_k}')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('K-Means - Silhouette Score', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. Fit Final Model

In [ ]:
km_final = KMeans(k=best_k, init='k-means++', n_init=10, random_state=42).fit(X)
labels = km_final.labels_
cents_orig = scaler.inverse_transform(km_final.centroids_)
print(f'Inertia: {km_final.inertia_:.2f}  Silhouette: {km_final.silhouette_score(X):.4f}')

## 5. Results and Visualisation

In [ ]:
cmap = plt.cm.get_cmap('tab10', best_k)
pairs = [(0,1,'Age','Annual Income (k$)'),(1,2,'Annual Income (k$)','Spending Score (1-100)'),(0,2,'Age','Spending Score (1-100)')]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (i, j, xl, yl) in zip(axes, pairs):
    for c in range(best_k):
        mask = labels==c
        ax.scatter(X_raw[mask,i], X_raw[mask,j], color=cmap(c), s=30, alpha=0.75, label=f'Cluster {c}')
    ax.scatter(cents_orig[:,i], cents_orig[:,j], marker='X', s=200, c='black', zorder=5)
    ax.set_xlabel(xl); ax.set_ylabel(yl); ax.set_title(f'{xl} vs {yl}', fontweight='bold')
handles, lbls = axes[0].get_legend_handles_labels()
fig.legend(handles, lbls, loc='lower center', ncol=best_k, frameon=False, bbox_to_anchor=(0.5,-0.05))
fig.suptitle(f'K-Means Clusters (k={best_k}) - Mall Customers', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

mall['Cluster'] = labels
print("\nCluster Profiles:")
print(mall.groupby('Cluster')[MALL_FEATURES].mean().round(1).to_string())

## 6. Analysis

**Best k=8 by silhouette score (0.428), Inertia=105.3**

The 8 clusters found represent distinct customer archetypes. From the cluster profile table:

| Cluster | Age | Income (k$) | Spending | Interpretation |
|---|---|---|---|---|
| 0 | 25 | 26 | 79 | Young, low income, high spenders — impulsive buyers |
| 1 | 41 | 89 | 16 | Middle-aged, wealthy, conservative spenders |
| 2 | 55 | 27 | 13 | Older, low income, low spenders — budget conscious |
| 3 | 25 | 56 | 50 | Young, mid income, average spenders |
| 4 | 65 | 53 | 50 | Older, mid income, average spenders |
| 5 | 33 | 87 | 82 | Young/mid, wealthy, high spenders — prime targets |
| 6 | 47 | 57 | 47 | Middle-aged, mid income, average spenders |
| 7 | 34 | 24 | 26 | Young, low income, low spenders |

**Clusters 5 and 0 are the most commercially valuable** — high spending scores regardless of income. Cluster 5 (young, high income, high spend) is especially attractive. Cluster 1 (wealthy but low spend) represents an opportunity: high-income customers who are not converting into high spenders.

**A silhouette score of 0.428** is moderate-to-good (the scale runs -1 to 1, with 0.5+ considered strong). The clusters are real but not perfectly separated — some customer types blend into each other.

**The three pairwise scatter plots** reveal that Income vs Spending Score gives the clearest visual separation, forming roughly five visible groupings. Age adds additional granularity but the clusters are less visually distinct along the age axis alone.

**Key takeaway:** K-Means successfully identifies actionable customer segments. The Income vs Spending Score plane is the most discriminating 2D view, and the high-income/high-spend segment (Cluster 5) is the obvious priority for targeted marketing.